# 05 — PCA & Dimensionality Reduction

# PCA – Reducción de dimensionalidad y visualización de clusters

## 1. Objetivo del notebook

En este notebook utilizaremos **PCA (Análisis de Componentes Principales)** para representar de forma visual los grupos de vinos obtenidos mediante el modelo de clustering.

El modelo de clustering trabaja con muchas variables al mismo tiempo. Como no podemos representar gráficamente tantas dimensiones, PCA las resume en dos nuevas variables:

- **PC1 (Componente Principal 1):** recoge la mayor cantidad posible de información de los datos.
- **PC2 (Componente Principal 2):** recoge la mayor cantidad posible de la información restante.

De esta forma, cada vino podrá representarse como un punto en un gráfico de dos dimensiones. El color del punto indicará el cluster al que pertenece.

### ¿Para qué utilizaremos PCA?

- Visualizar si los clusters están bien separados.
- Detectar grupos que se solapan.
- Identificar posibles vinos atípicos.
- Facilitar la interpretación y presentación de los resultados.

> **Importante:** PCA se aplicará sobre las mismas variables transformadas y escaladas utilizadas para entrenar el modelo de clustering. No utilizaremos directamente todas las columnas del CSV.

## 2. Importación de librerías

Importamos las herramientas que necesitaremos para:

- Cargar y manipular los datos.
- Aplicar PCA.
- Crear gráficos.
- Comprobar la calidad y la estructura de los datos.

In [1]:
# Librería para trabajar con datos organizados en filas y columnas
import pandas as pd

# Librería para realizar operaciones numéricas
import numpy as np

# Herramienta de scikit-learn para aplicar PCA
from sklearn.decomposition import PCA

# Librerías utilizadas para crear los gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración del estilo general de los gráficos
sns.set_theme(style="whitegrid")

# Mostramos todas las columnas del DataFrame cuando sea necesario
pd.set_option("display.max_columns", None)

## 3. Carga y comprobación inicial de los datos

Cargamos el archivo generado por `preprocessing.py`, que contiene los datos preparados para trabajar con los modelos.

Antes de aplicar PCA, comprobaremos:

- El número de filas y columnas.
- Los nombres de las columnas.
- Si existen valores nulos.
- Si hay filas duplicadas.

> Más adelante seleccionaremos únicamente las mismas variables que utilice el modelo de clustering.

In [2]:
# Ruta al dataset generado por el script de preprocesamiento.
# Al estar el notebook dentro de "notebooks", usamos ".." para volver
# a la raíz del proyecto y acceder a la carpeta "data/processed".
DATA_PATH = "../data/processed/wines_SPA_model_ready.csv"

# Cargamos la versión preparada para trabajar con los modelos.
# Este archivo se volverá a generar cuando se incorpore el nuevo preprocesamiento.
df = pd.read_csv(DATA_PATH)

# Primera comprobación visual de la estructura y los valores del dataset.
df.head()

,winery,wine_name,year,rating,region,price_euros,grape_variety,grape_variety_inferred,vine_type_inferred,wine_ageing,service_temperature,quality_price_ratio,luxury_category,flavor_descriptor,flavor_fruta_roja_negra,flavor_fruta_blanca_tropical,flavor_madera_especias,flavor_confiteria_tierra,flavor_floral_mineral,service_temp_label,service_temp_midpoint,region_mean_price,region_mean_rating,vine_type_Blanco,vine_type_Desconocido,vine_type_Espumoso,vine_type_Generoso,vine_type_Tinto,price_euros_scaled,rating_scaled,quality_price_ratio_scaled,year_scaled,region_mean_price_scaled,region_mean_rating_scaled
0,Teso La Monja,Tinto,2013,4.9,Toro,995.00,Tempranillo,1,0,0,16-18°C,0.005,1,"Cherry, Blackberry, Raspberry, Oak, Vanilla, L...",3,0,4,2,0,templado,17.0,208.907809,4.434694,0,0,0,0,1,3.054566,3.349969,-1.127575,0.162583,0.981804,0.681512
1,Artadi,Vina El Pison,2018,4.9,Vino de Espana,313.50,Tempranillo,0,1,0,16-18°C,0.016,1,"Cherry, Blackberry, Raspberry, Fig, Citrus, Or...",4,2,5,5,6,templado,17.0,184.233959,4.450908,0,0,0,0,1,0.618918,3.349969,-0.996195,0.614647,0.578428,1.102656
2,Vega Sicilia,Unico,2009,4.8,Ribera del Duero,324.95,Tempranillo,1,0,0,16-18°C,0.015,1,"Cherry, Blackberry, Plum, Fig, Peach, Apricot,...",4,7,6,4,7,templado,17.0,217.435097,4.456768,0,0,0,0,1,0.659840,2.673343,-1.008139,-0.199068,1.121211,1.254879
3,Vega Sicilia,Unico,1999,4.8,Ribera del Duero,692.96,Tempranillo,1,0,0,16-18°C,0.007,1,"Cherry, Blackberry, Plum, Fig, Peach, Apricot,...",4,7,6,4,7,templado,17.0,217.435097,4.456768,0,0,0,0,1,1.975090,2.673343,-1.103688,-1.103196,1.121211,1.254879
4,Vega Sicilia,Unico,1996,4.8,Ribera del Duero,778.06,Tempranillo,1,0,0,16-18°C,0.006,1,"Cherry, Blackberry, Plum, Fig, Peach, Apricot,...",4,7,6,4,7,templado,17.0,217.435097,4.456768,0,0,0,0,1,2.279233,2.673343,-1.115632,-1.374435,1.121211,1.254879


In [3]:
# Comprobamos el tamaño del dataset antes de seleccionar las variables del PCA.
# El primer número corresponde a las filas y el segundo a las columnas.
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

# Mostramos todos los nombres para conocer la estructura completa del archivo.
print("\nColumnas del dataset:")
for column in df.columns:
    print(f"- {column}")

Número de filas: 1918
Número de columnas: 34

Columnas del dataset:
- winery
- wine_name
- year
- rating
- region
- price_euros
- grape_variety
- grape_variety_inferred
- vine_type_inferred
- wine_ageing
- service_temperature
- quality_price_ratio
- luxury_category
- flavor_descriptor
- flavor_fruta_roja_negra
- flavor_fruta_blanca_tropical
- flavor_madera_especias
- flavor_confiteria_tierra
- flavor_floral_mineral
- service_temp_label
- service_temp_midpoint
- region_mean_price
- region_mean_rating
- vine_type_Blanco
- vine_type_Desconocido
- vine_type_Espumoso
- vine_type_Generoso
- vine_type_Tinto
- price_euros_scaled
- rating_scaled
- quality_price_ratio_scaled
- year_scaled
- region_mean_price_scaled
- region_mean_rating_scaled


## 4. Selección de variables para PCA

PCA se aplicará sobre la misma matriz de variables utilizada para entrenar el modelo de clustering.

De esta forma, la representación en dos dimensiones reflejará la información que el algoritmo ha utilizado realmente para formar los grupos.

> La selección definitiva de variables se completará cuando esté preparada la matriz final del clustering.